# PixCell Multi-Channel ControlNet — Colab Training (Stages 0–2)

This notebook runs the computationally expensive stages of the PixCell ControlNet pipeline on Google Colab:

| Stage | Script | What it does |
|-------|--------|--------------|
| **0** | `stage0_setup.py` | Download pretrained models (PixCell-256, ControlNet, UNI-2h, SD3.5 VAE) |
| **1** | `stage1_extract_features.py` | Extract UNI embeddings + VAE latents from H&E and cell-mask images |
| **2** | `stage2_train.py` | Train ControlNet + TME module on paired ORION-CRC data |

Stage 3 (inference) is lightweight and can run locally.

## Environment Setup

Import core libraries and set up the Colab environment.

In [ ]:
import os
import numpy as np
import torch
from IPython.display import clear_output
from google.colab import userdata

clear_output()

## Clone Repository & Install Dependencies

Clone the PixCell repo, check out the ControlNet branch, and install Python packages.

In [ ]:
%cd /content
!git clone https://github.com/pohaoc2/PixCell.git
%cd /content/PixCell
!git checkout main
!pip install -r requirements.txt
!pip install mmcv==1.7.0
clear_output()

## Configure AWS & HuggingFace Credentials

Set AWS credentials (for S3 data access) and HuggingFace token (for gated model downloads).
These should be stored in Colab's **Secrets** tab (`userdata`).

In [ ]:
!pip install awscli
clear_output()

os.environ["AWS_ACCESS_KEY_ID"] = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = "us-west-2"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## Download ORION-CRC Data from S3

Download the paired experimental dataset (H&E tiles + multi-channel TME maps) into `data/orion-crc/`.
The dataset contains:
- `he/` — H&E image tiles (256×256 PNG)
- `exp_channels/` — per-channel TME maps (cell_mask, cell_type_*, cell_state_*, vasculature, oxygen, glucose)

> **Note:** Update `S3_DATA_PATH` with the actual S3 URI before running.

In [ ]:
%cd /content/PixCell

S3_DATA_PATH = "s3://bagherilab-working/pohao/share_space/orion-crc"

!aws s3 sync {S3_DATA_PATH}/he          ./data/orion-crc/he/
!aws s3 sync {S3_DATA_PATH}/exp_channels ./data/orion-crc/exp_channels/
!aws s3 sync {S3_DATA_PATH}/features ./data/orion-crc/features/
!aws s3 sync {S3_DATA_PATH}/vae_features ./data/orion-crc/vae_features/

clear_output()
print("H&E tiles:", len(os.listdir("data/orion-crc/he")))
print("Channel dirs:", os.listdir("data/orion-crc/exp_channels"))

## Stage 0 — Download Pretrained Models

Download all four pretrained model components into `pretrained_models/`:
- **PixCell-256** transformer (frozen base model)
- **PixCell-256 ControlNet** (initialization weights)
- **UNI-2h** (histopathology feature extractor)
- **SD 3.5 VAE** (image encoder/decoder)

Requires `HF_TOKEN` for gated model access.

In [ ]:
%cd /content/PixCell
!python stage0_setup.py --token {os.environ['HF_TOKEN']}
clear_output()
print("Pretrained models:")
!ls pretrained_models/

## Stage 1 — Extract Features (Pass 1: H&E Images)

Extract **UNI-2h embeddings** (`*_uni.npy`) and **SD3.5 VAE latents** (`*_sd3_vae.npy`) from H&E tiles.
These are cached once and loaded at every training step.

In [ ]:
%cd /content/PixCell
!python stage1_extract_features.py \
    --image-dir  ./data/orion-crc/he \
    --output-dir ./data/orion-crc/features

## Stage 1 — Extract Features (Pass 2: Cell Masks)

Extract **VAE latents for cell_mask images** (`*_mask_sd3_vae.npy`).
These are used as additional ControlNet conditioning during training. UNI extraction is skipped for masks.

In [ ]:
%cd /content/PixCell
!python stage1_extract_features.py \
    --image-dir   ./data/orion-crc/exp_channels/cell_mask \
    --output-dir  ./data/orion-crc/vae_features \
    --vae-prefix  mask_sd3_vae \
    --skip-uni

## Stage 1 — Build HDF5 Metadata Index

Create the HDF5 index file that maps tile IDs to their channel/feature paths.
This index is loaded by the dataloader during training.

In [ ]:
from diffusion.data.datasets.paired_exp_controlnet_dataset import build_exp_index

build_exp_index(
    exp_channels_dir="data/orion-crc/exp_channels",
    output_path="data/orion-crc/metadata/exp_index.hdf5",
)

## Stage 2 — Train ControlNet

Train the ControlNet + TME conditioning module on paired ORION-CRC data.
Config is in `configs/config_controlnet_exp.py` (edit `exp_data_root`, `num_epochs`, etc. as needed).

Key training features:
- CFG dropout (15%) enables TME-only inference
- Channel reliability weights attenuate approximate channels (vasculature, oxygen, glucose)

In [ ]:
%cd /content/PixCell
!python stage2_train.py configs/config_controlnet_exp.py

## Upload Checkpoint to S3

Save the trained ControlNet checkpoint to S3 for later use in Stage 3 (inference).
Update the checkpoint path below to match the latest training step.

In [ ]:
ckpt_dir = "/content/PixCell/checkpoints/pixcell_controlnet_exp/checkpoints"
print("Available checkpoints:")
!ls {ckpt_dir}

ckpt_path = f"{ckpt_dir}/step_XXXXXXX"  # TODO: replace with actual step directory
!aws s3 cp {ckpt_path} s3://bagherilab-working/pohao/share_space/checkpoints/ --recursive

## Disconnect Runtime

Release the Colab GPU once training and upload are complete.

In [ ]:
from google.colab import runtime
runtime.unassign()